In [19]:
import sys
import requests
import pandas as pd
from bs4 import BeautifulSoup
import getpass
import time
from pathlib import Path


In [5]:
COMPANIES = {
    "Ford": {
        "ticker": "F",
        "cik": "0000037996"
    },
    "General Motors": {
        "ticker": "GM",
        "cik": "0001467858"
    },
    "Tesla": {
        "ticker": "TSLA",
        "cik": "0001318605"
    }
}

for company, details in COMPANIES.items():
    print(company, details)

Ford {'ticker': 'F', 'cik': '0000037996'}
General Motors {'ticker': 'GM', 'cik': '0001467858'}
Tesla {'ticker': 'TSLA', 'cik': '0001318605'}


In [6]:
def get_company_filings(cik):
    """
    Retrieve a company's recent SEC filing history.

    Parameters
    ----------
    cik : str
        The company's 10-digit SEC Central Index Key.

    Returns
    -------
    pandas.DataFrame
        Recent filings submitted by the company.
    """
    
    url = f"https://data.sec.gov/submissions/CIK{cik}.json"
    
    response = requests.get(
        url,
        headers=HEADERS,
        timeout=30
    )
    
    response.raise_for_status()
    
    company_data = response.json()
    recent_filings = company_data["filings"]["recent"]
    
    return pd.DataFrame(recent_filings)

In [7]:
contact_email = getpass.getpass(
    "Enter your email for the SEC User-Agent: "
)

HEADERS = {
    "User-Agent": f"CreditLens-RAG learning-project {contact_email}",
    "Accept-Encoding": "gzip, deflate",
    "Host": "data.sec.gov"
}

print("SEC request headers created.")

Enter your email for the SEC User-Agent:  ········


SEC request headers created.


In [9]:
ford_filings = get_company_filings(
    COMPANIES["Ford"]["cik"]
)

print("Number of recent Ford filings:", len(ford_filings))

ford_filings.head()

Number of recent Ford filings: 1001


,accessionNumber,filingDate,reportDate,acceptanceDateTime,act,form,fileNumber,filmNumber,items,core_type,size,isXBRL,isInlineXBRL,isXBRLNumeric,primaryDocument,primaryDocDescription
0,0000037996-26-000156,2026-07-29,2026-06-30,2026-07-28T19:29:21.000Z,34,10-Q,001-03950,261213398,,XBRL,14147761,1,1,1.0,f-20260630.htm,10-Q
1,0000037996-26-000155,2026-07-28,2026-07-28,2026-07-28T16:07:30.000Z,34,8-K,001-03950,261211889,"2.02,9.01",XBRL,5225266,1,1,0.0,f-20260728.htm,8-K
2,0000037996-26-000151,2026-07-02,2026-07-02,2026-07-02T09:16:14.000Z,34,8-K,001-03950,261148221,"8.01,9.01",XBRL,1632913,1,1,0.0,f-20260702.htm,8-K
3,0000037996-26-000149,2026-06-24,2026-06-23,2026-06-24T16:16:30.000Z,,4,,,,4,5267,0,0,0.0,xslF345X06/wk-form4_1782332186.xml,FORM 4
4,0000037996-26-000146,2026-06-05,2026-06-04,2026-06-05T16:29:48.000Z,,4,,,,4,8565,0,0,0.0,xslF345X06/wk-form4_1780691385.xml,FORM 4


In [10]:
ford_10k = (
    ford_filings[ford_filings["form"] == "10-K"]
    [
        [
            "filingDate",
            "reportDate",
            "accessionNumber",
            "primaryDocument"
        ]
    ]
    .head(3)
    .reset_index(drop=True)
)

ford_10k

,filingDate,reportDate,accessionNumber,primaryDocument
0,2026-02-11,2025-12-31,0000037996-26-000015,f-20251231.htm
1,2025-02-06,2024-12-31,0000037996-25-000013,f-20241231.htm
2,2024-02-07,2023-12-31,0000037996-24-000009,f-20231231.htm


In [11]:
all_10k_filings = []

for company_name, company_info in COMPANIES.items():
    
    filings = get_company_filings(company_info["cik"])
    
    latest_10k = (
        filings[filings["form"] == "10-K"]
        [
            [
                "filingDate",
                "reportDate",
                "accessionNumber",
                "primaryDocument"
            ]
        ]
        .head(3)
        .copy()
    )
    
    latest_10k["company"] = company_name
    latest_10k["ticker"] = company_info["ticker"]
    latest_10k["cik"] = company_info["cik"]
    
    all_10k_filings.append(latest_10k)
    
    print(f"Retrieved {len(latest_10k)} filings for {company_name}")
    
    time.sleep(0.2)

Retrieved 3 filings for Ford
Retrieved 3 filings for General Motors
Retrieved 3 filings for Tesla


In [12]:
filings_df = pd.concat(
    all_10k_filings,
    ignore_index=True
)

filings_df = filings_df[
    [
        "company",
        "ticker",
        "cik",
        "filingDate",
        "reportDate",
        "accessionNumber",
        "primaryDocument"
    ]
]

filings_df

,company,ticker,cik,filingDate,reportDate,accessionNumber,primaryDocument
0,Ford,F,0000037996,2026-02-11,2025-12-31,0000037996-26-000015,f-20251231.htm
1,Ford,F,0000037996,2025-02-06,2024-12-31,0000037996-25-000013,f-20241231.htm
2,Ford,F,0000037996,2024-02-07,2023-12-31,0000037996-24-000009,f-20231231.htm
3,General Motors,GM,0001467858,2026-01-27,2025-12-31,0001467858-26-000013,gm-20251231.htm
4,General Motors,GM,0001467858,2025-01-28,2024-12-31,0001467858-25-000032,gm-20241231.htm
5,General Motors,GM,0001467858,2024-01-30,2023-12-31,0001467858-24-000031,gm-20231231.htm
6,Tesla,TSLA,0001318605,2026-01-29,2025-12-31,0001628280-26-003952,tsla-20251231.htm
7,Tesla,TSLA,0001318605,2025-01-30,2024-12-31,0001628280-25-003063,tsla-20241231.htm
8,Tesla,TSLA,0001318605,2024-01-29,2023-12-31,0001628280-24-002390,tsla-20231231.htm


In [13]:
filings_df.shape

(9, 7)

In [14]:
HEADERS.pop("Host", None)

'data.sec.gov'

In [15]:
def build_filing_url(row):
    cik_without_zeros = str(int(row["cik"]))
    accession_without_hyphens = row["accessionNumber"].replace("-", "")
    
    return (
        "https://www.sec.gov/Archives/edgar/data/"
        f"{cik_without_zeros}/"
        f"{accession_without_hyphens}/"
        f"{row['primaryDocument']}"
    )

In [16]:
filings_df["filing_url"] = filings_df.apply(
    build_filing_url,
    axis=1
)

filings_df[
    [
        "company",
        "reportDate",
        "filing_url"
    ]
]

,company,reportDate,filing_url
0,Ford,2025-12-31,https://www.sec.gov/Archives/edgar/data/37996/...
1,Ford,2024-12-31,https://www.sec.gov/Archives/edgar/data/37996/...
2,Ford,2023-12-31,https://www.sec.gov/Archives/edgar/data/37996/...
3,General Motors,2025-12-31,https://www.sec.gov/Archives/edgar/data/146785...
4,General Motors,2024-12-31,https://www.sec.gov/Archives/edgar/data/146785...
5,General Motors,2023-12-31,https://www.sec.gov/Archives/edgar/data/146785...
6,Tesla,2025-12-31,https://www.sec.gov/Archives/edgar/data/131860...
7,Tesla,2024-12-31,https://www.sec.gov/Archives/edgar/data/131860...
8,Tesla,2023-12-31,https://www.sec.gov/Archives/edgar/data/131860...


In [17]:
print(filings_df.loc[0, "filing_url"])

https://www.sec.gov/Archives/edgar/data/37996/000003799626000015/f-20251231.htm


In [18]:
test_response = requests.get(
    filings_df.loc[0, "filing_url"],
    headers=HEADERS,
    timeout=30
)

print("Status code:", test_response.status_code)
print("Downloaded size:", len(test_response.content), "bytes")

Status code: 200
Downloaded size: 5281502 bytes


In [20]:
def find_repository_root(start_path=Path.cwd()):
    """
    Search the current directory and its parents
    for the creditlens-rag Git repository.
    """
    
    for folder in [start_path, *start_path.parents]:
        if (folder / ".git").exists():
            return folder
    
    raise FileNotFoundError(
        "Could not locate the Git repository."
    )


REPO_ROOT = find_repository_root()
RAW_DATA_DIR = REPO_ROOT / "data" / "raw"

RAW_DATA_DIR.mkdir(parents=True, exist_ok=True)

print("Repository:", REPO_ROOT)
print("Download location:", RAW_DATA_DIR)

Repository: D:\analytics\A_Python_Code\creditlens-rag
Download location: D:\analytics\A_Python_Code\creditlens-rag\data\raw


In [21]:
downloaded_files = []

for index, row in filings_df.iterrows():
    
    company = row["company"]
    ticker = row["ticker"]
    reporting_year = row["reportDate"][:4]
    filing_url = row["filing_url"]
    
    file_name = f"{ticker}_{reporting_year}_10K.html"
    file_path = RAW_DATA_DIR / file_name
    
    try:
        response = requests.get(
            filing_url,
            headers=HEADERS,
            timeout=60
        )
        
        response.raise_for_status()
        file_path.write_bytes(response.content)
        
        file_size_mb = file_path.stat().st_size / (1024 * 1024)
        
        downloaded_files.append({
            "company": company,
            "ticker": ticker,
            "reporting_year": reporting_year,
            "file_name": file_name,
            "file_size_mb": round(file_size_mb, 2),
            "status": "Downloaded"
        })
        
        print(
            f"Downloaded: {file_name} "
            f"({file_size_mb:.2f} MB)"
        )
        
    except requests.RequestException as error:
        downloaded_files.append({
            "company": company,
            "ticker": ticker,
            "reporting_year": reporting_year,
            "file_name": file_name,
            "file_size_mb": None,
            "status": f"Failed: {error}"
        })
        
        print(f"Failed: {file_name}")
        print(error)
    
    time.sleep(0.2)

Downloaded: F_2025_10K.html (5.04 MB)
Downloaded: F_2024_10K.html (5.25 MB)
Downloaded: F_2023_10K.html (5.27 MB)
Downloaded: GM_2025_10K.html (3.93 MB)
Downloaded: GM_2024_10K.html (3.96 MB)
Downloaded: GM_2023_10K.html (3.85 MB)
Downloaded: TSLA_2025_10K.html (2.28 MB)
Downloaded: TSLA_2024_10K.html (2.48 MB)
Downloaded: TSLA_2023_10K.html (2.55 MB)


In [22]:
download_results_df = pd.DataFrame(downloaded_files)

download_results_df

,company,ticker,reporting_year,file_name,file_size_mb,status
0,Ford,F,2025,F_2025_10K.html,5.04,Downloaded
1,Ford,F,2024,F_2024_10K.html,5.25,Downloaded
2,Ford,F,2023,F_2023_10K.html,5.27,Downloaded
3,General Motors,GM,2025,GM_2025_10K.html,3.93,Downloaded
4,General Motors,GM,2024,GM_2024_10K.html,3.96,Downloaded
5,General Motors,GM,2023,GM_2023_10K.html,3.85,Downloaded
6,Tesla,TSLA,2025,TSLA_2025_10K.html,2.28,Downloaded
7,Tesla,TSLA,2024,TSLA_2024_10K.html,2.48,Downloaded
8,Tesla,TSLA,2023,TSLA_2023_10K.html,2.55,Downloaded


In [23]:
successful_downloads = (
    download_results_df["status"] == "Downloaded"
).sum()

print("Successful downloads:", successful_downloads)
print("Expected downloads:", len(filings_df))

Successful downloads: 9
Expected downloads: 9


In [24]:
html_files = list(RAW_DATA_DIR.glob("*.html"))

print("HTML files found:", len(html_files))

for file in sorted(html_files):
    print(file.name)

HTML files found: 9
F_2023_10K.html
F_2024_10K.html
F_2025_10K.html
GM_2023_10K.html
GM_2024_10K.html
GM_2025_10K.html
TSLA_2023_10K.html
TSLA_2024_10K.html
TSLA_2025_10K.html


In [25]:
metadata_path = REPO_ROOT / "data" / "filings_metadata.csv"

filings_df.to_csv(
    metadata_path,
    index=False
)

print("Metadata saved to:", metadata_path)

Metadata saved to: D:\analytics\A_Python_Code\creditlens-rag\data\filings_metadata.csv
